# AME end-to-end example (genriesz)

This notebook demonstrates how to estimate an **Average Marginal Effect (AME)**,
i.e., an **average derivative** of the outcome regression function.

We simulate

$$
Y = \sin(X_0) + 0.5 X_1^2 + \varepsilon,
$$

so the true AME for coordinate 0 is

$$
\mathbb{E}[\partial_{x_0} \gamma(X)] = \mathbb{E}[\cos(X_0)].
$$

If $X_0 \sim N(0,1)$, then $\mathbb{E}[\cos(X_0)] = \exp(-1/2)$.


In [1]:
import numpy as np
from genriesz import (
    grr_ame,
    SquaredGenerator,
    PolynomialBasis,
    RBFRandomFourierBasis,
)

rng = np.random.default_rng(0)

## Synthetic data with known true AME

In [2]:
n = 4000
d = 3

X = rng.normal(size=(n, d))
eps = rng.normal(scale=1.0, size=n)

Y = np.sin(X[:, 0]) + 0.5 * (X[:, 1] ** 2) + eps

true_ame0 = float(np.exp(-0.5))  # E[cos(N(0,1))]
print("Approx. true AME for coordinate 0:", true_ame0)


Approx. true AME for coordinate 0: 0.6065306597126334


## Example 1: Polynomial basis

In [3]:
# A simple polynomial basis on X
basis = PolynomialBasis(degree=3, include_bias=True)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_ame(
    X=X,
    Y=Y,
    coordinate=0,
    basis=basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())


AME(coord=0) estimates (n=4000)
alpha=0.05 | null=0.0
diagnostics: alpha_abs_mean=0.7986210631930721, alpha_abs_p95=1.9651976737301404, alpha_abs_max=4.506477411795792, riesz_modifies_estimand=False, riesz_fit_success_rate=1.0, riesz_gradient_norm_max=4.301485721975619e-15, riesz_kkt_residual_max=4.301485721975619e-15, riesz_clip_binding_rate_max=0.0, held_out_imbalance_max=0.6008320243780351, held_out_imbalance_mean=0.08979565706509104, bias_proxy=0.030321403103141308, std_bias=1.7878744714734351, outcome_cv_risk=0.9932197515491971, outcome_residual_var=0.993467771839262

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                0.568023    0.00659669       [ 0.555094,  0.580952]           0
RW                0.570009     0.0255017       [ 0.520027,  0.619992]   1.16e-110
ARW               0.568095     0.0169595       [ 0.534855,  0.601335]   5.29e-246
TMLE      

## Example 2: RKHS basis (RBF random Fourier features)

RBF random Fourier features are smooth and differentiable, so the AME derivative
``d phi / d x_j`` is implemented analytically.

In [4]:
psi_rff = RBFRandomFourierBasis(
    n_features=500,
    sigma=1.0,
    standardize=True,
    random_state=0,
)

res_rff = grr_ame(
    X=X,
    Y=Y,
    coordinate=0,
    basis=psi_rff,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rff.summary_text())

AME(coord=0) estimates (n=4000)
alpha=0.05 | null=0.0
diagnostics: alpha_abs_mean=0.73277748961463, alpha_abs_p95=1.8122057162097691, alpha_abs_max=2.4279560645811697, riesz_modifies_estimand=False, riesz_fit_success_rate=1.0, riesz_gradient_norm_max=2.8327492261615017e-16, riesz_kkt_residual_max=2.8327492261615017e-16, riesz_clip_binding_rate_max=0.0, held_out_imbalance_max=0.026829812665075604, held_out_imbalance_mean=0.001730349826389184, bias_proxy=0.037106776579452175, std_bias=2.364915068579797, outcome_cv_risk=1.0450958329886644, outcome_residual_var=1.045332829304062

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                0.548985      0.007015       [ 0.535236,  0.562734]           0
RW                0.539443     0.0197649       [ 0.500705,  0.578182]   5.14e-164
ARW               0.564788     0.0156905        [ 0.534035,  0.59554]   9.86e-284
TMLE   

## Note: bases requiring smooth derivatives

AME requires ``basis.derivative(X, coordinate)``.  **Piecewise-constant** bases
(``KNNCatchmentBasis``, ``RandomForestLeafBasis``, ``TorchEmbeddingBasis`` without
autograd) do not implement this method and therefore cannot be used with ``grr_ame``.
Use ``PolynomialBasis`` or ``RBFRandomFourierBasis`` (or any smooth basis) instead.

## Generator sweep (SQ / UKL / BP)

Below we compare SQ-Riesz, UKL-Riesz, and BP-Riesz under multiple regularization
norms and strengths.  We report **RA / RW / ARW / TMLE** and the error against
the known true AME.

In [5]:
# A small grid over generators and regularization.
# Branchwise generators (UKL, BP) require a branch selector fixed by the
# estimand. The AME representer changes sign with x, so no such branch exists
# and this example uses the squared generator only.
generator_grid = [
    ("SQ", SquaredGenerator(C=0.0).as_generator()),
]

penalty_grid = [
    {"penalty": "l2", "lam": 1e-4, "p_norm": None},
    {"penalty": "l2", "lam": 1e-3, "p_norm": None},
    {"penalty": "l1", "lam": 1e-4, "p_norm": None},
    {"penalty": "lp", "lam": 1e-3, "p_norm": 1.5},
]

rows = []
for gname, gen_i in generator_grid:
    for cfg in penalty_grid:
        res_i = grr_ame(
            X=X,
            Y=Y,
            coordinate=0,
            basis=basis,
            generator=gen_i,
            cross_fit=True,
            folds=3,               # smaller folds for the sweep
            random_state=0,
            estimators=("ra", "rw", "arw", "tmle"),
            outcome_models="shared",
            outcome_link="identity",  # Y is unbounded, so Gaussian TMLE is appropriate
            riesz_penalty=cfg["penalty"],
            riesz_lam=cfg["lam"],
            riesz_p_norm=cfg.get("p_norm"),
            max_iter=250,
            tol=1e-8,
        )

        row = {
            "generator": gname,
            "penalty": cfg["penalty"],
            "lam": cfg["lam"],
        }

        for k in ("ra", "rw", "arw", "tmle"):
            e = res_i.estimates[k]
            row[f"{k}"] = e.estimate
            row[f"{k}_se"] = e.se
            row[f"{k}_err"] = e.estimate - true_ame0

        rows.append(row)

import pandas as pd

df = pd.DataFrame(rows)
# Sort by absolute ARW error (ARW is typically stable)
df = df.sort_values(by="arw_err", key=lambda s: np.abs(s))
display(df)

,generator,penalty,lam,ra,ra_se,ra_err,rw,rw_se,rw_err,arw,arw_se,arw_err,tmle,tmle_se,tmle_err
3,SQ,lp,0.0010,0.56645,0.006685,-0.040081,0.575628,0.026159,-0.030903,0.569669,0.017074,-0.036862,0.569543,0.017104,-0.036987
1,SQ,l2,0.0010,0.56645,0.006685,-0.040081,0.575372,0.026199,-0.031158,0.569663,0.017072,-0.036868,0.569537,0.017102,-0.036993
0,SQ,l2,0.0001,0.56645,0.006685,-0.040081,0.576793,0.026183,-0.029738,0.569646,0.017092,-0.036884,0.569518,0.017121,-0.037013
2,SQ,l1,0.0001,0.56645,0.006685,-0.040081,0.576778,0.026169,-0.029752,0.569638,0.017091,-0.036893,0.569510,0.017121,-0.037021
